In [1]:
import pandas as pd
import os

#Ici on vient créer les dossiers si jamais il y'a besoin
os.makedirs('../inputs', exist_ok=True)       
os.makedirs('../outputs', exist_ok=True)   

annees = [2018, 2019, 2020, 2021, 2022]
liste_df = []

for annee in annees:
    print("Traitement de l'année", annee)
    chemin = f'../inputs/base-flux-mobilite-residentielle-{annee}.csv'
    
    #Lecture des fichiers csv 
    df = pd.read_csv(chemin, sep=';', dtype={'CODGEO': str, 'DCRAN': str})
    nom_col_flux = ""
    for col in df.columns:
        if "NBFLUX" in col:
            nom_col_flux = col
            
    #On renomme pour + de clarté
    df = df.rename(columns={
        'CODGEO': 'code_dest',
        'LIBGEO': 'nom_dest',
        'DCRAN': 'code_orig',
        'L_DCRAN': 'nom_orig',
        nom_col_flux: 'flux'
    })
    df['annee'] = annee
    df = df[~df['code_orig'].str.startswith('99', na=False)] #les flux venant de l'étranger (99) sont supprimés
    df = df[df['code_dest'] != df['code_orig']] #Les personnes qui déménagent dans la même ville sont supprimés
    df = df.dropna(subset=['flux']) #On enlève les cases vides ou = à zéro
    df = df[df['flux'] > 0]
    
    df_clean = df[['annee', 'code_orig', 'nom_orig', 'code_dest', 'nom_dest', 'flux']]
    liste_df.append(df_clean)

#On fusionne...
print("Fusion...")
df_final = pd.concat(liste_df, ignore_index=True)

chemin_export = '../outputs/flux_migratoire_triee.csv'
df_final.to_csv(chemin_export, index=False, sep=';')
print("Terminé, le fichier est sauvegardé dans le dossier 'outputs'")

Traitement de l'année 2018
Traitement de l'année 2019
Traitement de l'année 2020
Traitement de l'année 2021
Traitement de l'année 2022
Fusion...
Terminé, le fichier est sauvegardé dans le dossier 'outputs'
